In [ ]:
!pip install torch transformers accelerate chromadb sentence-transformers networkx numpy scikit-learn


In [ ]:
import os
import numpy as np
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from chromadb.config import Settings
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
EMBED_MODEL = "all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TINYLLAMA_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHROMA_DIR = "./graph_rag_db"
GRAPH_SIM_THRESHOLD = 0.65
TOP_K_RETRIEVE = 5
TOP_K_EXPANDED = 20
RERANK_TOP_K = 5

In [ ]:
# -----------------------------------
# Step 1: Create a large document on Lebesgue Integration
# -----------------------------------
document = """
# The Lebesgue Integral: A Deep Dive into Modern Integration Theory

The Lebesgue integral is a fundamental concept in modern analysis and probability theory.
It extends the Riemann integral to a broader class of functions and provides a more powerful framework
for studying convergence, measure, and functional spaces.

## 1. Motivation and Limitations of the Riemann Integral
The Riemann integral, introduced in the 19th century, approximates area under a curve by partitioning the x-axis into intervals and summing rectangles.
However, it fails for functions that are discontinuous on dense subsets, or when dealing with limits of sequences of functions that behave irregularly.
The Riemann approach depends heavily on the partition of the domain rather than the values the function takes.

For instance, consider the indicator function of rational numbers on [0, 1]:
f(x) = 1 if x is rational, 0 otherwise.
This function is not Riemann integrable because rationals are dense but have measure zero — yet the function “jumps” everywhere.
Lebesgue’s idea was to reverse the logic: integrate by measuring where the function takes certain values.

## 2. The Measure-Theoretic Foundation
At the heart of the Lebesgue integral lies **measure theory**.
Measure theory generalizes the notions of length, area, and volume.

A **measure space** is a triple (X, Σ, μ), where:
- X is the set under consideration (e.g., real numbers ℝ),
- Σ is a σ-algebra (collection of measurable subsets of X),
- μ is a measure, assigning a non-negative real number to each set in Σ.

The **Lebesgue measure** on ℝ generalizes the intuitive notion of length.
For example, μ([a, b]) = b - a, but the measure can also handle highly irregular sets.

## 3. Simple Functions and Stepwise Construction
The Lebesgue integral is first defined for **simple functions** — those that take only finitely many values.

If φ = ∑ a_i χ_{A_i}, where χ_{A_i} is the indicator of a measurable set A_i,
then the integral of φ over a set E is:
∫_E φ dμ = ∑ a_i μ(E ∩ A_i)

For nonnegative measurable functions f, the Lebesgue integral is defined as the supremum of integrals of simple functions that are less than or equal to f.

This stepwise construction ensures the definition works even for functions that are not continuous or bounded.

## 4. Integration of General Functions
For real-valued functions that can take positive and negative values, we define:
f⁺ = max(f, 0)
f⁻ = max(-f, 0)
and require that ∫ f⁺ dμ and ∫ f⁻ dμ are both finite.
Then,
∫ f dμ = ∫ f⁺ dμ - ∫ f⁻ dμ.

This decomposition ensures the Lebesgue integral handles signed functions while maintaining convergence properties.

## 5. Dominated and Monotone Convergence Theorems
The Lebesgue integral shines through its **powerful convergence theorems**,
which make it the standard in modern analysis.

### (a) Monotone Convergence Theorem (MCT)
If {f_n} is an increasing sequence of nonnegative measurable functions converging pointwise to f, then:
∫ f_n dμ → ∫ f dμ.

### (b) Dominated Convergence Theorem (DCT)
If {f_n} → f pointwise and there exists an integrable function g such that |f_n| ≤ g for all n,
then:
∫ f_n dμ → ∫ f dμ.

These results fail under Riemann integration but hold under Lebesgue integration, enabling powerful limit operations.

## 6. Relationship with the Riemann Integral
Every Riemann integrable function on [a, b] is also Lebesgue integrable, and both integrals coincide in value.
However, the converse is not true: there exist Lebesgue integrable functions that are not Riemann integrable.

The Lebesgue integral extends the concept to a broader space L¹(μ), the set of absolutely integrable functions.

## 7. Lᵖ Spaces and Functional Analysis
Lebesgue integration leads naturally to the concept of **Lᵖ spaces**:
Lᵖ(X, μ) = { f measurable | ∫ |f|ᵖ dμ < ∞ }.

These spaces are central in functional analysis, partial differential equations, and probability theory.
- L¹: absolutely integrable functions
- L²: square-integrable functions, forming a Hilbert space
- L∞: essentially bounded functions

Inner products and norms in these spaces enable geometric reasoning about functions.

## 8. Applications in Probability and Statistics
In probability theory, random variables are measurable functions on a probability space.
The expected value E[X] is precisely a Lebesgue integral:
E[X] = ∫ X dP,
where P is a probability measure.

This abstraction allows for defining continuous distributions, stochastic processes, and conditional expectations rigorously.

## 9. Examples and Computations
### Example 1: Indicator Function
Let f(x) = χ_[0, 1/2](x) on [0, 1].
Then ∫ f dμ = μ([0, 1/2]) = 1/2.

### Example 2: Unbounded Function
f(x) = 1/√x on [0, 1].
Lebesgue integral converges since ∫₀¹ 1/√x dx = 2, though the function is unbounded near 0.

### Example 3: Non-Riemann Integrable but Lebesgue Integrable
f(x) = χ_ℚ(x) on [0, 1].
Lebesgue integral = 0 because ℚ has measure zero.

## 10. Comparison Summary

| Property | Riemann Integral | Lebesgue Integral |
|-----------|------------------|------------------|
| Domain partition | Based on intervals | Based on function values |
| Handles discontinuities | Poorly | Very well |
| Limit theorems | Limited | Strong (MCT, DCT, Fatou) |
| Integrable set | Continuous or piecewise smooth | Any measurable, integrable function |
| Foundation | Geometry (area under curve) | Measure theory |

## 11. Extensions: Product Measures and Fubini’s Theorem
The Lebesgue integral generalizes beautifully to multiple dimensions.
For product spaces (X × Y, μ × ν),
Fubini’s theorem allows changing the order of integration:
∫∫ f(x, y) dμ(x) dν(y) = ∫∫ f(x, y) dν(y) dμ(x),
under mild conditions of integrability.

This theorem is foundational for multidimensional calculus, probability, and differential equations.

## 12. Beyond Lebesgue: Abstract Integration
Modern extensions include:
- **Radon–Nikodym theorem**: Relating one measure to another via a derivative.
- **Stieltjes integral**: Integration with respect to non-uniform measures.
- **Bochner integral**: Extending integration to vector-valued functions.

These build upon the same ideas introduced by Lebesgue in the early 20th century.

# Summary
The Lebesgue integral unifies geometry, measure, and convergence into a single, elegant framework.
It replaces ad hoc restrictions of Riemann integration with a robust theory that powers much of modern mathematics — from analysis to probability and functional spaces.
Its introduction marks one of the greatest conceptual advances in the history of mathematical analysis.
"""


In [ ]:
def chunk_text(text, chunk_size=60, overlap=15):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(document)
print(f"Document split into {len(chunks)} chunks.")


Document split into 24 chunks.


In [ ]:
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
embeddings = embedder.encode(chunks)
print("Embeddings generated.")


Loading embedding model...
Embeddings generated.


In [ ]:
print("Setting up Chroma...")
client = chromadb.PersistentClient(path=CHROMA_DIR)

try:
    client.delete_collection("graph_rag_collection")
    print("Existing collection deleted.")
except:
    print("No existing collection to delete.")

collection = client.create_collection("graph_rag_collection")

ids = [f"chunk_{i}" for i in range(len(chunks))]
collection.add(ids=ids, documents=chunks, embeddings=embeddings.tolist())

# No need for client.persist() — PersistentClient auto-saves
print(f"{len(chunks)} chunks added to Chroma.")


Setting up Chroma...
Existing collection deleted.
24 chunks added to Chroma.


In [ ]:
G = nx.Graph()
for i, text in enumerate(chunks):
    G.add_node(ids[i], text=text, emb=embeddings[i])

cos_sim = cosine_similarity(embeddings)
for i in range(len(chunks)):
    for j in range(i+1, len(chunks)):
        if cos_sim[i][j] > GRAPH_SIM_THRESHOLD:
            G.add_edge(ids[i], ids[j], weight=float(cos_sim[i][j]))

print(f"Graph created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph created: 24 nodes, 17 edges


In [ ]:
def retrieve_chunks(query, top_k=TOP_K_RETRIEVE):
    q_emb = embedder.encode([query])[0].tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    retrieved = []
    for i, cid in enumerate(results["ids"][0]):
        retrieved.append({"id": cid, "text": results["documents"][0][i]})
    return retrieved

query = "What is the main difference between the Riemann and Lebesgue integral?"
seed_chunks = retrieve_chunks(query)
print("Initial retrieved chunks:")
for s in seed_chunks:
    print("-", s["id"], ":", s["text"][:80])


Initial retrieved chunks:
- chunk_12 : powerful limit operations. ## 6. Relationship with the Riemann Integral Every Ri
- chunk_0 : # The Lebesgue Integral: A Deep Dive into Modern Integration Theory The Lebesgue
- chunk_3 : if x is rational, 0 otherwise. This function is not Riemann integrable because r
- chunk_1 : and functional spaces. ## 1. Motivation and Limitations of the Riemann Integral 
- chunk_2 : or when dealing with limits of sequences of functions that behave irregularly. T


In [ ]:
def expand_graph(seed_ids, hops=1, max_candidates=TOP_K_EXPANDED):
    candidates = set(seed_ids)
    frontier = set(seed_ids)
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            for neighbor in G.neighbors(node):
                candidates.add(neighbor)
                next_frontier.add(neighbor)
        frontier = next_frontier
    expanded = [G.nodes[c]["text"] for c in list(candidates)[:max_candidates]]
    return expanded

seed_ids = [s["id"] for s in seed_chunks]
expanded_chunks = expand_graph(seed_ids)
print(f"Expanded to {len(expanded_chunks)} chunks using graph.")

Expanded to 8 chunks using graph.


In [ ]:
print("Reranking candidates...")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
pairs = [[query, text] for text in expanded_chunks]
scores = cross_encoder.predict(pairs)

ranked = sorted(zip(expanded_chunks, scores), key=lambda x: x[1], reverse=True)
top_texts = [t for t, _ in ranked[:RERANK_TOP_K]]
print("Top reranked chunks ready.")

Reranking candidates...
Top reranked chunks ready.


In [ ]:
context = "\n\n".join(top_texts)
prompt = f"Use only the context below to answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"


In [ ]:
print("Loading TinyLlama model (local)...")
tokenizer = AutoTokenizer.from_pretrained(TINYLLAMA_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    TINYLLAMA_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def get_answer_local(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.split("Answer:")[-1].strip()

final_answer = get_answer_local(prompt)
print("\n=== FINAL ANSWER ===\n")
print(final_answer)

Loading TinyLlama model (local)...

=== FINAL ANSWER ===

The main difference between the Riemann and Lebesgue integral is that the Lebesgue integral extends the concept to a broader class of functions, while the Riemann integral only applies to Riemann integrable functions.
